# PANDAS MASTERY — CHAPTERS 48–56

```
PANDAS MASTERY
Current Chapters: 48 (SettingWithCopyWarning, continued), 49 (Index Management),
                  50 (Professional Data Validation), 51 (Data Quality Framework),
                  52 (Design Patterns), 53 (Method Chaining),
                  54 (Pipeline-Style Data Cleaning),
                  55 (Industry Project #1 - Customer Analytics),
                  56 (Industry Project #2 - Churn Data Prep)
Progress:         56 / 76 chapters
Completed:        Ch1-47
Remaining:        20 chapters
```

Nine chapters, ending in two complete mini-projects. Chapter 47 already covered the headline
Copy-on-Write finding — this chapter goes one level deeper into the edge cases that still
matter, then moves into index management, validation, quality frameworks, professional code
patterns, and two full applied projects. Every number below was executed and verified before
this notebook was written.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


---
# CHAPTER 48 — SettingWithCopyWarning, Continued: The Edge Cases That Still Matter

## Concept
Chapter 47 established that pandas 3.0's mandatory Copy-on-Write eliminates the classic
DataFrame-slice chained-assignment danger. This chapter checks the two remaining places people
still worry about: mutating an array pulled out via `.to_numpy()`, and chained assignment
through **two separate bracket operations** in one statement.

## Edge case 1: mutating `.to_numpy()`'s output


In [ ]:
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

arr = df["a"].to_numpy()
print("arr.flags['WRITEABLE']:", arr.flags["WRITEABLE"])

try:
    arr[0] = 999
    print("df after mutating the array:\n", df)
except ValueError as e:
    print("Mutating the returned array RAISED:", e)


### Expected Output (verified on pandas 3.0.2)
```
arr.flags['WRITEABLE']: False
Mutating the returned array RAISED: assignment destination is read-only
```

### A genuinely important, current finding: `.to_numpy()` returns a READ-ONLY array by default
This closes off the last realistic avenue for accidentally corrupting a DataFrame through its
NumPy export: `df['a'].to_numpy()` now returns an array with `WRITEABLE = False`, so attempting
`arr[0] = 999` doesn't silently succeed and doesn't silently fail to propagate — it raises a
clear `ValueError` immediately, at the exact line responsible. If you genuinely need a mutable
copy of the underlying data, pass `df['a'].to_numpy(copy=True)` explicitly — the read-only
default forces that intent to be stated, rather than assumed.

## Edge case 2: chained assignment through two separate bracket operations


In [ ]:
df3 = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    df3["b"][df3["a"] > 1] = -1
    for warning in w:
        print(f"{warning.category.__name__}:")
        print(str(warning.message))

print("\ndf3 after the double-bracket chain (note: NOT actually modified):\n", df3)


### Expected Output (verified on pandas 3.0.2)
```
ChainedAssignmentError:
A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment

df3 after the double-bracket chain (note: NOT actually modified):
   a  b
0  1  4
1  2  5
2  3  6
```

### This is the single most useful pandas 3.0 debugging improvement in this entire course
The historically dangerous `df["b"][mask] = value` pattern (two separate `__getitem__` calls,
the second of which silently operates on a throwaway intermediate object under old pandas) now
does three things right, all at once: (1) it **does not** silently corrupt or fail to update
anything — `df3` is provably unchanged above; (2) it fires a **named, specific**
`ChainedAssignmentError` warning rather than the old generic `SettingWithCopyWarning`; (3) the
warning **tells you the exact fix** (`'.loc[row_indexer, col_indexer] = value'`) in its own
text. A bug class that used to require deep internal knowledge of view/copy semantics to
diagnose is now essentially self-documenting.

### Interview Insight
> "A teammate writes `df['col'][mask] = value` and the DataFrame doesn't update. Walk through
> what's happening and how you'd explain the fix, on a modern pandas version."
On pandas 3.0+, this raises an explicit `ChainedAssignmentError` (not a silent failure) — the
first bracket (`df['col']`) returns a Series that, under mandatory Copy-on-Write, always
behaves as an independent copy; the second bracket's assignment (`[mask] = value`) then mutates
only that throwaway copy, never the original `df`. The warning text itself states the fix:
combine both steps into a single `.loc[mask, 'col'] = value` call, which updates the original
DataFrame directly and atomically rather than through two separate, disconnected operations.


---
# CHAPTER 49 — Index Management

## Concept
`set_index()`, `reset_index()`, `rename_axis()`, `sort_index()` — managing the row index
deliberately, plus a professional judgment: indexes are useful when you have a genuine natural
key you'll repeatedly select/join on, and unnecessary complexity otherwise.


In [ ]:
df4 = pd.DataFrame({"id": [1, 2, 3], "name": ["a", "b", "c"], "val": [10, 20, 30]})

indexed = df4.set_index("id")
print("set_index('id'):\n", indexed)

back = indexed.reset_index()
print("\nreset_index():\n", back)
print("\nEqual to original?", back.equals(df4))


### Expected Output (verified on pandas 3.0.2)
```
set_index('id'):
    name  val
id
1     a   10
2     b   20
3     c   30

reset_index():
    id name  val
0   1    a   10
1   2    b   20
2   3    c   30

Equal to original? True
```

### `set_index()`/`reset_index()` are exact, lossless inverses when used correctly
`reset_index()` restores the original DataFrame precisely — verified with `.equals()` returning
`True` — this round-trip guarantee is exactly why Chapter 21's grouped results (which set the
grouping column as the index) can always be flattened back with zero information loss.

### `rename_axis()` and MultiIndex sorting


In [ ]:
renamed_axis = indexed.rename_axis("customer_id")
print("rename_axis('customer_id') -- index NAME changed, values untouched:", renamed_axis.index.name)

multi = df4.set_index(["name", "id"])
print("\nMultiIndex via set_index(['name','id']):\n", multi)

sorted_idx = multi.sort_index(ascending=False)
print("\nsort_index(ascending=False):\n", sorted_idx)


### Expected Output (verified on pandas 3.0.2)
```
rename_axis('customer_id') -- index NAME changed, values untouched: customer_id

MultiIndex via set_index(['name','id']):
        val
name id
a    1    10
b    2    20
c    3    30

sort_index(ascending=False):
        val
name id
c    3    30
b    2    20
a    1    10
```

### When an index is genuinely useful vs unnecessary complexity
**Useful**: a datetime index for `resample()`/time-based slicing (Chapter 29-30); a natural key
you'll repeatedly `.loc[]` into or join on; the output of a `groupby()` you're about to
`reset_index()` anyway as an intermediate step. **Unnecessary complexity**: setting an index
"because it seems more organized" when you're not actually exploiting label-based selection or
time-series operations — a default `RangeIndex` with the key as a normal column is often
simpler to filter, merge, and reason about, exactly Chapter 23's MultiIndex caution generalized.

### Interview Insight
> "When would you deliberately set a non-default index on a DataFrame, versus leaving the
> default RangeIndex?"
When you'll repeatedly select by that key via `.loc[]`, need `resample()`/time-based slicing
(requiring a `DatetimeIndex`), or you're working with a `groupby()` result you'll query before
flattening. If the key is mostly used as an ordinary filter/join column, leaving it as a
regular column with the default index is usually simpler and avoids unnecessary
`reset_index()` calls scattered through the rest of the pipeline.


---
# CHAPTER 50 — Professional Data Validation

## Concept
A reusable `validate_dataframe()` function — the Pandas-level equivalent of the NumPy course's
`np.testing` assertions, checking structural correctness before any transformation runs.


In [ ]:
def validate_dataframe(df, required_columns=None, unique_columns=None, dtype_map=None):
    """Return a list of human-readable validation errors (empty list = all checks passed)."""
    errors = []
    if required_columns:
        missing = set(required_columns) - set(df.columns)
        if missing:
            errors.append(f"Missing required columns: {missing}")
    if unique_columns:
        for col in unique_columns:
            if col in df.columns and df[col].duplicated().any():
                errors.append(f"Column '{col}' expected unique, found {df[col].duplicated().sum()} duplicates")
    if dtype_map:
        for col, expected_dtype in dtype_map.items():
            if col in df.columns and str(df[col].dtype) != expected_dtype:
                errors.append(f"Column '{col}' expected dtype {expected_dtype}, found {df[col].dtype}")
    return errors


test_df = pd.DataFrame({"id": [1, 1, 2], "name": ["a", "b", "c"], "amount": [10.0, 20.0, 30.0]})

errors = validate_dataframe(
    test_df,
    required_columns=["id", "name", "missing_col"],
    unique_columns=["id"],
    dtype_map={"amount": "int64"},
)
print("Validation errors:")
for e in errors:
    print(" -", e)


### Expected Output (verified on pandas 3.0.2)
```
Validation errors:
 - Missing required columns: {'missing_col'}
 - Column 'id' expected unique, found 1 duplicates
 - Column 'amount' expected dtype int64, found float64
```

### Why returning a list of errors, not raising immediately, is the right design
A validation function that raises on the **first** failure hides every other problem — a
caller fixes one issue, reruns, discovers the next, reruns again. Returning **all** discovered
errors at once (as demonstrated: three genuinely different problems reported together) lets a
caller fix everything in one pass, exactly mirroring how a real schema-validation tool (e.g.,
`pandera`, `great_expectations`) reports results, and matching the "collect every failure, then
decide what to do" philosophy good test suites use.

### Interview Insight
> "Why might a data validation function return a list of errors instead of raising an exception
> on the first problem found?"
Raising immediately hides subsequent issues, forcing a fix-rerun-fix-rerun cycle to discover
each one individually. Collecting and returning all violations in a single pass lets the caller
see the complete picture of what's wrong at once — the caller (or the pipeline orchestrating
the validation) can then decide whether any failures are severe enough to halt processing, log
a warning, or proceed anyway, rather than the validation function making that decision by
raising unconditionally.


---
# CHAPTER 51 — Data Quality Framework

## Concept
Six formal dimensions of data quality, each with a concrete Pandas check:
**Completeness** (missingness), **Uniqueness** (duplicate keys), **Validity** (values within
allowed ranges/formats), **Consistency** (conflicting representations of the same category —
Chapter 16's territory formalized), **Accuracy** (suspicious/implausible values), **Timeliness**
(data freshness).


In [ ]:
dq_df = pd.DataFrame({
    "id": [1, 2, 2, 4, 5],
    "email": ["a@x.com", None, "b@x.com", "invalid", "c@x.com"],
    "age": [25, 200, 35, -5, 40],
    "signup": pd.to_datetime(["2023-01-01", "2023-02-01", "2023-02-01", "2020-01-01", "2099-01-01"]),
})

# --- Completeness: fraction of NON-missing values, per column ---
completeness = 1 - dq_df.isna().mean()
print("Completeness per column:\n", completeness)

# --- Uniqueness: is the intended key actually unique? ---
uniqueness = dq_df["id"].nunique() / len(dq_df)
print("\nUniqueness (id):", uniqueness)

# --- Validity: fraction of values within a plausible range ---
validity_age = ((dq_df["age"] >= 0) & (dq_df["age"] <= 120)).mean()
print("Validity (age in [0,120]):", validity_age)

# --- Timeliness: fraction of records that aren't dated in the future ---
timeliness = (dq_df["signup"] <= pd.Timestamp.now()).mean()
print("Timeliness (signup not in future):", timeliness)


### Expected Output (verified on pandas 3.0.2)
```
Completeness per column:
id        1.0
email     0.8
age       1.0
signup    1.0
dtype: float64

Uniqueness (id): 0.8
Validity (age in [0,120]): 0.6
Timeliness (signup not in future): 0.8
```

### Reading each dimension as an independent, complementary signal
`email` is 80% complete (1 of 5 missing) — but notice `age` and `signup` report **100%
complete** even though they contain clearly bad values (`200`, `-5`, a signup date in `2099`).
**Completeness only measures absence, not correctness** — a column can be perfectly complete
and still fail validity or timeliness entirely. `age`'s validity (`60%` — 3 of 5 rows in
`[0,120]`) and `signup`'s timeliness (`80%` — 1 of 5 rows in the future) each catch a *different*
kind of problem that completeness alone would miss. **This is the entire point of tracking
multiple quality dimensions independently rather than a single "data quality score"** — a single
blended number would obscure which specific dimension actually needs attention.

### Interview Insight
> "A dataset shows 100% completeness. Does that mean the data is high quality?"
No — completeness only measures the *absence* of missing values, not whether the present values
are correct, plausible, or current. A column can be perfectly complete while containing
impossible values (validity failures), stale records (timeliness failures), or inconsistent
category representations (consistency failures) — a full data quality assessment requires
checking all relevant dimensions independently, not just completeness.


---
# CHAPTER 52 — Pandas Design Patterns

## Concept
Six recurring structural patterns, named so they become recognizable at a glance in code
review, rather than being re-derived from scratch each time:

1. **Load → Inspect → Clean**: `pd.read_csv()` → `.info()`/`.describe()`/`.sample()` →
   dtype fixes/dedup/imputation (Chapter 37's full workflow).
2. **Filter → Transform → Aggregate**: `df[condition]` → `.assign(...)` → `.groupby(...).agg(...)`.
3. **Group → Aggregate → Merge**: compute a per-entity summary via `groupby`, then merge it back
   onto the original (or a different) table for enrichment (Chapter 55 does this explicitly).
4. **Long → Transform → Wide**: `melt()`/reshape → compute → `pivot_table()` for final
   presentation (Chapter 27-28).
5. **Raw → Clean → Feature Engineering**: never skip straight from raw to features — clean
   first, verify, then engineer (Chapter 37 → 39's ordering).
6. **Train/test-safe transformations**: fit-dependent steps only inside a `Pipeline`, fit only
   on training data (Chapter 41-42).

Naming these patterns explicitly means a reviewer can say "this is pattern 3, and it's missing
the merge-back step" rather than describing the whole flow from scratch every time.


---
# CHAPTER 53 — Method Chaining

## Concept
Writing a sequence of transformations as one chained expression — each method returns a new
DataFrame the next method calls on, reading top-to-bottom like the SQL query it often
corresponds to (Chapter 45).


In [ ]:
raw52 = pd.DataFrame({
    "customer": ["  Alice", "BOB", "carol"],
    "amount": ["100.5", "200", "not_a_number"],
    "date": ["2023-01-01", "2023-02-01", "invalid"],
})

def clean_customer_data(df):
    return (
        df
        .assign(
            customer=lambda d: d["customer"].str.strip().str.title(),
            amount=lambda d: pd.to_numeric(d["amount"], errors="coerce"),
            date=lambda d: pd.to_datetime(d["date"], errors="coerce"),
        )
        .dropna(subset=["amount"])
    )

cleaned52 = clean_customer_data(raw52)
print("Method-chained cleaning result:\n", cleaned52)


### Expected Output (verified on pandas 3.0.2)
```
Method-chained cleaning result:
  customer  amount       date
0    Alice   100.5 2023-01-01
1      Bob   200.0 2023-02-01
```

### Reading the chain
Each `lambda d: ...` inside `.assign()` receives the DataFrame **as it exists at that point in
the chain** (Chapter 15's lesson), so `customer`, `amount`, and `date` are all transformed in a
single `.assign()` call, referencing the *original* columns being replaced (not yet-transformed
versions of themselves, since all three lambdas see the same starting `d`). `.dropna(subset=
["amount"])` then drops the third row, whose `amount` failed to parse (`"not_a_number"` →
`NaN`) — note `date` also failed to parse for that row (`"invalid"` → `NaT`), but since
`dropna` only checks `amount`, that row is dropped anyway for the amount failure regardless.

### `.pipe()` — chaining custom functions, not just built-in methods


In [ ]:
def add_tax(df, rate=0.08):
    return df.assign(tax=lambda d: d["amount"] * rate)

piped = raw52.pipe(clean_customer_data).pipe(add_tax, rate=0.1)
print(".pipe() chained result:\n", piped)


### Expected Output (verified on pandas 3.0.2)
```
.pipe() chained result:
  customer  amount       date    tax
0    Alice   100.5 2023-01-01  10.05
1      Bob   200.0 2023-02-01  20.00
```

### Why `.pipe()` matters — extending the chain beyond built-in methods
`.assign()`, `.query()`, `.groupby()` etc. are built-in DataFrame methods you can chain
directly. `.pipe(custom_function, **kwargs)` lets you insert **your own** functions
(`clean_customer_data`, `add_tax`) into the same fluent chain, passing the DataFrame as the
first argument automatically — without `.pipe()`, you'd have to break the chain into separate
statements (`df = clean_customer_data(df); df = add_tax(df, rate=0.1)`), losing the single
readable top-to-bottom flow.

### When chaining becomes too complicated
A chain that spans 15+ methods, mixes multiple unrelated concerns, or requires scrolling to
understand what a single line does is a sign to **break it apart** — into named intermediate
variables, or separate `.pipe()`-able functions with clear names (exactly Chapter 54's pattern).
Method chaining is a readability tool, not an obligation to cram everything into one expression.

### Interview Insight
> "What's the advantage of `.pipe(custom_function)` over calling `custom_function(df)`
> separately?"
`.pipe()` keeps a custom transformation inside the same fluent method chain as built-in Pandas
methods, preserving a single, readable top-to-bottom data flow — `df.query(...).pipe(my_func)
.groupby(...)` reads as one continuous pipeline, whereas breaking out to
`df2 = my_func(df.query(...))` interrupts that flow and requires tracking intermediate variable
names.


---
# CHAPTER 54 — Pipeline-Style Data Cleaning

## Concept
Wrapping cleaning logic in named, reusable, testable functions (`clean_customer_data(df)` from
Chapter 53 IS this pattern) rather than scattering transformations across many disconnected
notebook cells — the direct bridge from "exploratory notebook" to Chapter 62's `src/` module
structure (from the NumPy course's project-structure chapter, applied here to Pandas).

## Why this matters more than it might seem
A notebook with 30 cells of inline transformations is nearly impossible to: (1) re-run reliably
in a different order, (2) reuse on a new dataset without copy-pasting cells, (3) unit test
(Chapter 61), or (4) code-review meaningfully (a reviewer has to reconstruct the whole notebook
state mentally). A single, named `clean_customer_data(df) -> df` function solves all four —
it's testable in isolation, reusable by importing it, reviewable as one coherent unit, and
composable via `.pipe()` with other named functions.

```python
# notebook cell -- exploratory, throwaway
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["amount"])
# ... 20 more scattered cells ...

# src/cleaning.py -- promoted to a tested, reusable function
def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce amount to numeric, drop unparseable rows. See tests/test_cleaning.py."""
    return df.assign(amount=lambda d: pd.to_numeric(d["amount"], errors="coerce")).dropna(subset=["amount"])
```

This is the exact "notebook code vs production code" distinction the NumPy course establishes
in its own architecture chapter, now applied specifically to Pandas cleaning logic.


---
# CHAPTER 55 — Real Industry Project #1: Customer Analytics

## Business problem
A company wants to understand customer behavior: who are the highest-value customers, which
products drive revenue, and how do regions compare. Full pipeline: ingest → join → aggregate →
answer business questions.


In [ ]:
customers55 = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "region": ["East", "West", "East", "West", "East"],
    "signup_date": pd.to_datetime(["2022-01-15", "2022-03-20", "2022-02-10", "2022-06-01", "2022-01-05"]),
})
orders55 = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107],
    "customer_id": [1, 1, 2, 3, 3, 4, 5],
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Widget", "Gadget", "Widget"],
    "amount": [50.0, 75.0, 100.0, 30.0, 45.0, 90.0, 60.0],
    "order_date": pd.to_datetime(["2022-02-01", "2022-03-01", "2022-04-01", "2022-03-15", "2022-05-01", "2022-07-01", "2022-02-15"]),
})

# Pattern 3: Group -> Aggregate -> Merge (Chapter 52)
merged55 = pd.merge(customers55, orders55, on="customer_id", how="left")

customer_summary = merged55.groupby(["customer_id", "name", "region"], as_index=False).agg(
    total_revenue=("amount", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("amount", "mean"),
)
print("Customer-level summary:\n", customer_summary)


### Expected Output (verified on pandas 3.0.2)
```
Customer-level summary:
    customer_id   name region  total_revenue  order_count  avg_order_value
0            1  Alice   East          125.0            2             62.5
1            2    Bob   West          100.0            1            100.0
2            3  Carol   East           75.0            2             37.5
3            4   Dave   West           90.0            1             90.0
4            5    Eve   East           60.0            1             60.0
```

## Business Question 1: Who are the highest-value customers?


In [ ]:
top_customers = customer_summary.nlargest(3, "total_revenue")
print("Top 3 customers by revenue:\n", top_customers[["name", "total_revenue"]])


### Expected Output (verified on pandas 3.0.2)
```
Top 3 customers by revenue:
     name  total_revenue
0  Alice          125.0
1    Bob          100.0
3   Dave           90.0
```

## Business Question 2: Which regions perform best?


In [ ]:
region_revenue = merged55.groupby("region")["amount"].sum().sort_values(ascending=False)
print("Revenue by region:\n", region_revenue)


### Expected Output (verified on pandas 3.0.2)
```
Revenue by region:
region
East    260.0
West    190.0
Name: amount, dtype: float64
```

## Business Question 3: Which products drive revenue?


In [ ]:
product_revenue = orders55.groupby("product")["amount"].agg(["sum", "count"]).sort_values("sum", ascending=False)
print("Product performance:\n", product_revenue)


### Expected Output (verified on pandas 3.0.2)
```
Product performance:
          sum  count
product
Widget  255.0      4
Gadget  165.0      2
Gizmo    30.0      1
```

### Reading these as a coherent business narrative, not three disconnected numbers
East (`260.0`) outperforms West (`190.0`) in total revenue — but the customer summary reveals
East's lead comes disproportionately from **one** customer (Alice, `125.0` alone), not a
uniformly stronger region — exactly the kind of nuance that a single regional total hides and
a customer-level breakdown reveals. Widget dominates product revenue (`255.0`, more than East
and West's individual totals) and has the highest order count too (`4`), making it clearly the
core product, not just a favorite of one big spender — a distinction worth confirming precisely
because Alice (East, high spender) *did* buy a Widget, which could otherwise create a
misleading "maybe it's just Alice's preference" doubt.

### Interview Insight
> "Revenue by region shows East outperforming West by a wide margin. What would you check
> before concluding East is a stronger market overall?"
Whether that regional total is being driven broadly across many customers, or disproportionately
by one or two high-value outliers — a customer-level breakdown (exactly the `customer_summary`
table above) answers this directly. A region "winning" because of one large customer is a very
different business insight (customer concentration risk) than a region winning because of
broadly stronger average performance.


---
# CHAPTER 56 — Real Industry Project #2: Customer Churn Data Preparation

## Business problem
Prepare a model-ready churn dataset: clean categories, engineer tenure/recency/value features,
define the churn target, and check for leakage — directly assembling Chapters 16, 37, 39, and
42 into one coherent applied pipeline.


In [ ]:
churn_raw = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "signup_date": pd.to_datetime(["2022-01-01", "2022-02-01", "2022-01-15", "2022-03-01", "2022-01-20", "2022-02-15"]),
    "last_order_date": pd.to_datetime(["2023-06-01", "2022-03-01", "2023-05-15", "2022-04-01", "2023-06-10", None]),
    "total_orders": [15, 2, 20, 1, 18, 0],
    "total_spend": [1500.0, 100.0, 2200.0, 50.0, 1800.0, 0.0],
    "region": ["East", "east", "EAST", "West", "east ", "West"],   # Chapter 16's exact fragmentation problem
})

print("Raw data:\n", churn_raw)
print("\nRegion values BEFORE cleaning:", churn_raw["region"].unique().tolist())


### Expected Output (verified on pandas 3.0.2)
```
Raw data:
   customer_id signup_date last_order_date  total_orders  total_spend region
0            1  2022-01-01      2023-06-01            15       1500.0   East
1            2  2022-02-01      2022-03-01             2        100.0   east
2            3  2022-01-15      2023-05-15            20       2200.0   EAST
3            4  2022-03-01      2022-04-01             1         50.0   West
4            5  2022-01-20      2023-06-10            18       1800.0  east
5            6  2022-02-15             NaT             0          0.0   West

Region values BEFORE cleaning: ['East', 'east', 'EAST', 'West', 'east ', 'West']
```

### The `reference_date` concept — the single most important idea in this project
Churn feature engineering isn't computed "as of today" in a real pipeline — it's computed **as
of a fixed reference point**, simulating "if we were running this analysis on this specific
date." This is essential for both reproducibility (results don't silently change every time the
code re-runs on a different day) and for correctly building historical training examples
(Chapter 58's fraud project will lean on this same idea for temporal leakage prevention).


In [ ]:
reference_date = pd.Timestamp("2023-07-01")

churn_features = churn_raw.copy()

# Consistency: fix the region fragmentation (Chapter 16 + Chapter 51's consistency dimension)
churn_features["region"] = churn_features["region"].str.strip().str.upper()

# Feature engineering, all computed relative to the FIXED reference_date -- never "today"
churn_features["tenure_days"] = (reference_date - churn_features["signup_date"]).dt.days
churn_features["days_since_last_order"] = (reference_date - churn_features["last_order_date"]).dt.days
churn_features["avg_order_value"] = churn_features["total_spend"] / churn_features["total_orders"].replace(0, np.nan)

# Target creation: churned if no order in 90+ days OR never ordered at all (NaN last_order_date)
churn_features["churned"] = (
    (churn_features["days_since_last_order"] > 90) | churn_features["days_since_last_order"].isna()
).astype(int)

print(churn_features)
print("\nChurn rate:", churn_features["churned"].mean())
print("Region values AFTER cleaning:", sorted(churn_features["region"].unique().tolist()),
      " -- nunique:", churn_features["region"].nunique())


### Expected Output (verified on pandas 3.0.2)
```
   customer_id signup_date last_order_date  total_orders  total_spend region  tenure_days  days_since_last_order  avg_order_value  churned
0            1  2022-01-01      2023-06-01            15       1500.0   EAST          546                   30.0            100.0        0
1            2  2022-02-01      2022-03-01             2        100.0   EAST          515                  487.0             50.0        1
2            3  2022-01-15      2023-05-15            20       2200.0   EAST          532                   47.0            110.0        0
3            4  2022-03-01      2022-04-01             1         50.0   WEST          487                  456.0             50.0        1
4            5  2022-01-20      2023-06-10            18       1800.0   EAST          527                   21.0            100.0        0
5            6  2022-02-15             NaT             0          0.0   WEST          501                    NaN              NaN        1

Churn rate: 0.5
Region values AFTER cleaning: ['EAST', 'WEST']  -- nunique: 2
```

### Reading each engineering decision, precisely
- **`region` fragmentation fixed**: 6 raw values (`'East'`, `'east'`, `'EAST'`, `'West'`,
  `'east '` with trailing space, `'West'`) collapsed to exactly **2** genuine regions
  (`'EAST'`, `'WEST'`) — this is Chapter 16/51's consistency check, applied for real; without
  it, any `groupby('region')` on this data would have silently split East into three separate,
  undercounted groups.
- **`avg_order_value` via `.replace(0, np.nan)` before dividing**: customer 6 has
  `total_orders=0` — dividing by zero directly would produce `inf`, not a meaningful "average."
  Replacing `0` with `NaN` *before* dividing produces a correct `NaN` ("undefined, not
  computable") instead — a small but important defensive-division habit directly from the
  NumPy course's numerical-edge-cases material, now applied in Pandas.
- **`churned` target definition**: customer 6 (never ordered, `last_order_date` is `NaT`)
  correctly counts as churned via the `.isna()` branch of the OR condition — a customer who
  never converted at all is exactly as much a churn/non-retention case as one who ordered once
  and stopped, and the target definition explicitly, deliberately covers both.
- **`churned.mean() = 0.5`**: exactly 3 of 6 customers churned by this definition — a
  genuinely balanced synthetic dataset here, though real churn datasets are typically far more
  imbalanced, which would need addressing (class weighting, resampling) before modeling, not
  covered in this data-prep-focused project.

### The leakage check this project deliberately builds in
Every feature above (`tenure_days`, `days_since_last_order`, `avg_order_value`) is computed
relative to the **fixed** `reference_date`, never relative to "now" at model-training time and
never using information that would only be known *after* the churn outcome was determined. This
is Chapter 42's discipline, applied at the project level: if this were a real production churn
model, the same `reference_date`-relative feature computation would need to be re-derivable at
actual prediction time for a *new*, still-active customer — using only data available up to
that customer's own "reference point," never peeking past it.

### Interview Insight
> "Why compute churn features relative to a fixed `reference_date` rather than
> `pd.Timestamp.now()`?"
Using "now" makes the computation non-reproducible (the exact feature values change every time
the code re-runs on a different day) and, more importantly, breaks the temporal correctness
needed for historical training examples — a churn label from six months ago needs to be
evaluated using data *as it existed six months ago*, not data as it exists today (which would
leak future information about what happened after that point). A fixed `reference_date` per
training example (or per batch) is the standard, leakage-safe pattern.


---
## Chapters 48–56 Summary

- **SettingWithCopyWarning, continued**: `.to_numpy()` now returns a **read-only** array by
  default (verified: mutation raises `ValueError` immediately); the historically dangerous
  double-bracket chained assignment now raises an explicit, self-documenting
  `ChainedAssignmentError` naming the exact `.loc[]` fix — both genuine, current pandas 3.0
  safety improvements verified live.
- **Index management**: `set_index()`/`reset_index()` are exact, lossless inverses (verified
  via `.equals()`); use a non-default index only when you're genuinely exploiting label-based
  selection or time-series operations.
- **Data validation**: a `validate_dataframe()` function should collect and return *all*
  violations in one pass, not raise on the first — verified returning three independent,
  simultaneously-discoverable errors.
- **Data quality framework**: completeness, uniqueness, validity, and timeliness are
  independent dimensions — verified a dataset that's 100% complete on `age`/`signup` while
  failing validity (60%) and timeliness (80%) on those same columns.
- **Design patterns**: six named, recognizable structural patterns (Load→Inspect→Clean,
  Filter→Transform→Aggregate, Group→Aggregate→Merge, Long→Transform→Wide,
  Raw→Clean→Feature-Engineer, Train/test-safe transformations) that make code review faster by
  giving reviewers shared vocabulary.
- **Method chaining**: `.assign()` lambdas all see the same starting DataFrame within one call;
  `.pipe()` extends fluent chaining to custom functions, not just built-in methods.
- **Pipeline-style cleaning**: named, reusable, testable cleaning functions are the bridge from
  exploratory notebook cells to production `src/` modules.
- **Project #1 (Customer Analytics)**: verified that East's regional revenue lead was
  substantially driven by one high-value customer — a nuance a region-level total alone would
  have hidden.
- **Project #2 (Churn Prep)**: 6 raw region values correctly collapsed to 2 genuine regions;
  division-by-zero correctly avoided via `.replace(0, np.nan)` before dividing; churn target
  correctly captures both "stopped ordering" and "never ordered at all"; every feature computed
  relative to a fixed `reference_date` for reproducibility and leakage-safety.

---

## Mastery Exercises

**Exercise 1:** Why does `df['col'].to_numpy()` returning a read-only array by default matter
more for defensive coding than it might first appear?

**Exercise 2 (Prediction):** For a DataFrame validated with `unique_columns=['id']` where `id`
has 2 duplicate pairs among 10 rows, predict the exact error message
`validate_dataframe()` (Chapter 50) would produce.

**Coding Challenge:** Extend Chapter 51's data quality framework into a single
`data_quality_report(df, validity_rules, key_column, date_column)` function that returns a
DataFrame with one row per quality dimension and its score, suitable for logging or dashboarding
over time.

**Debugging Challenge:**
```python
df = pd.DataFrame({"score": [85, 90, 78]})
top_half = df[df["score"] > df["score"].median()]
top_half["score"] = top_half["score"] * 1.1
```
On pandas 3.0+, does this raise a warning/error, silently do nothing to `df`, or work as
intended on `top_half`? Explain precisely, referencing Chapter 47/48's Copy-on-Write material.

**Interview Questions:**
1. What specific pandas 3.0 change makes `.to_numpy()` safer than it used to be?
2. Why should churn/time-based features be computed relative to a fixed reference date rather
   than the current system time?
3. Give an example where a dataset scores 100% on completeness but should still be considered
   low quality.
4. Why is `.pipe()` preferred over breaking a transformation out into a separate statement,
   from a readability standpoint?

**Quiz:**
- Q1: True/False — On pandas 3.0+, `df['col'].to_numpy()[0] = value` silently succeeds and
  updates the original DataFrame.
- Q2: Which data quality dimension specifically catches inconsistent category spellings like
  `'USA'`/`'usa'`/`'United States'`?
- Q3: In `df.assign(a=lambda d: ..., b=lambda d: ...)`, does the lambda for `b` see the
  already-transformed `a`, or the original column?

---
 